# Asistente RAG — Mesa de Ayuda de TI (Instituto Tecnológico Cóndor)

**Sistemas Inteligentes Computacionales · Parcial práctico RAG (Corte 2)**

Asistente que responde preguntas sobre una colección de documentos de una mesa de ayuda de TI, **cita la evidencia** (archivo + localizador) y **se abstiene** cuando la evidencia es insuficiente.

> El corpus es **sintético**: la institución "Instituto Tecnológico Cóndor" es ficticia y todos los documentos fueron generados para este proyecto (`herramientas/generar_corpus.py`). No contiene datos reales ni personales, así que es apto para procesarse con servicios externos (OpenRouter).

## Cómo ejecutar (Google Colab)
1. **Clave de OpenRouter:** en Colab, abre el panel 🔑 *Secretos* (barra izquierda), crea el secreto `OPENROUTER_API_KEY` con tu clave y activa el acceso para este cuaderno. **Nunca escribas la clave en una celda.**
2. **Archivos del proyecto:** el cuaderno necesita las carpetas `corpus/` y `evaluacion/`. En la celda 1 elige una opción: clonar el repositorio (`REPO_URL`) o subir `proyecto_rag.zip` con la carpeta del proyecto. Si falta `corpus/`, se regenera con `herramientas/generar_corpus.py`.
3. Ejecuta **Entorno de ejecución → Ejecutar todas**. Las corridas finales (que gastan llamadas al LLM) solo se ejecutan si pones `EJECUTAR_CORRIDAS = True` en la sección 9.

## Flujo
```
corpus/ (PDF, DOCX, MD) → ingesta con localizadores → fragmentación (R0 fija 400/60 · R1 fija 200/30)
   → embeddings (e5-small · bge-m3 en B) → índice coseno → recuperación top-k (densa · híbrida opcional)
   → contexto con límite de tokens → prompt con citas [F#] / abstención → LLM (OpenRouter)
   → respuesta + fragmentos en orden + citas (archivo · localizador) → registro de corridas → métricas
```

## Configuraciones
| Config | Qué cambia respecto a R0 | Experimento |
|---|---|---|
| **R0** | Base: fragmentos fijos de 400 tokens con 60 de solapamiento, coseno, k = 4 | A |
| **R1** | Solo la fragmentación: 200 tokens con 30 de solapamiento (misma proporción, 15 %) | A |
| **R2** | Solo la recuperación: k = 8 | A |
| **C0** | Sin recuperación: mismo LLM, sin documentos, prompt propio, sin citas | A |
| **B_EMB** | Otro modelo de embeddings (índice reconstruido y consultas recodificadas) | B |
| **B_LLM** | Otro LLM | B |
| R2_HIBRIDA | (Opcional) densa + BM25 con fusión RRF, k = 4 | — |

## 1. Preparación del entorno

In [1]:
import os, sys, subprocess, zipfile
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

# --- Opción A: clonar el repositorio del equipo (deja "" si no lo usan) ---
REPO_URL = ""          # p. ej. "https://github.com/usuario/proyecto-rag.git"
# --- Opción B: subir un zip con la carpeta del proyecto (corpus/, evaluacion/, herramientas/) ---
NOMBRE_ZIP = "proyecto_rag.zip"

def _buscar_raiz():
    for c in [Path.cwd(), *Path.cwd().parents, Path("/content/proyecto_rag")]:
        if (c / "evaluacion" / "preguntas.json").exists():
            return c
    return None

RAIZ = _buscar_raiz()
if RAIZ is None and EN_COLAB:
    if REPO_URL:
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/proyecto_rag"], check=True)
    else:
        from google.colab import files
        print(f"Sube {NOMBRE_ZIP} (zip de la carpeta del proyecto)…")
        subidos = files.upload()
        with zipfile.ZipFile(next(iter(subidos))) as z:
            z.extractall("/content/proyecto_rag")
    RAIZ = _buscar_raiz()
    if RAIZ is None:  # el zip pudo traer una carpeta intermedia
        candidatos = list(Path("/content/proyecto_rag").rglob("evaluacion/preguntas.json"))
        RAIZ = candidatos[0].parent.parent if candidatos else None
assert RAIZ is not None, "No encuentro la carpeta del proyecto (debe contener evaluacion/preguntas.json)."
os.chdir(RAIZ)
print("Carpeta del proyecto:", RAIZ)

Carpeta del proyecto: /home/jullian/Documentos/2026_2/Sistemas_inteligentes/Proyecto_Parcial


In [2]:
# Dependencias (en Colab se instalan; en local usa requirements.txt)
if EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pypdf", "python-docx", "sentence-transformers", "rank_bm25", "openai", "fpdf2"], check=True)

# Si falta el corpus, se regenera (es sintético y determinista)
if not (RAIZ / "corpus").exists() or not any((RAIZ / "corpus").iterdir()):
    subprocess.run([sys.executable, "herramientas/generar_corpus.py"], check=True)

In [3]:
import json, re, time, unicodedata, hashlib, platform, getpass
from datetime import datetime, timezone
from dataclasses import dataclass, field, asdict

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", 120)

# Clave de OpenRouter: secreto de Colab → variable de entorno → .env (local) → pedirla sin mostrarla
def _cargar_clave():
    if os.environ.get("OPENROUTER_API_KEY"):
        return os.environ["OPENROUTER_API_KEY"]
    if EN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get("OPENROUTER_API_KEY")
        except Exception:
            pass
    env = RAIZ / ".env"
    if env.exists():
        for linea in env.read_text().splitlines():
            if linea.startswith("OPENROUTER_API_KEY="):
                return linea.split("=", 1)[1].strip().strip('"')
    return None

OPENROUTER_API_KEY = _cargar_clave()
if not OPENROUTER_API_KEY and EN_COLAB:
    OPENROUTER_API_KEY = getpass.getpass("OPENROUTER_API_KEY (no se mostrará): ") or None
print("Clave de OpenRouter cargada:", "sí" if OPENROUTER_API_KEY else "NO (las celdas del LLM fallarán)")

Clave de OpenRouter cargada: NO (las celdas del LLM fallarán)


## 2. Parámetros y configuraciones

Todo lo que se mantiene fijo entre configuraciones está aquí: modelos, temperatura, límite de salida, límite de contexto y plantillas. **No cambien estos valores después de congelar las hipótesis.**

- **Unidad de tamaño:** tokens del tokenizador de `intfloat/multilingual-e5-small` (el modelo de embeddings base). Los fragmentos se calculan siempre con ese tokenizador, así que en `B_EMB` los fragmentos son idénticos a R0 y solo cambian los vectores.
- **Límite de contexto:** 2400 tokens (mismo tokenizador). Con k = 4 × 400 no recorta; con k = 8 × 400 (R2) obliga a recortar, y eso queda registrado.
- **Modelos del LLM:** los IDs gratuitos de OpenRouter cambian con el tiempo. La celda de la sección 6 verifica que existan y, si no, lista alternativas.

In [4]:
EMB_PRINCIPAL   = "intfloat/multilingual-e5-small"   # 384 dim, máx. 512 tokens, prefijos query:/passage:
EMB_ALTERNATIVO = "BAAI/bge-m3"                      # experimento B (1024 dim, máx. 8192 tokens)
LLM_PRINCIPAL   = "google/gemma-4-31b-it:free"
LLM_ALTERNATIVO = "nvidia/nemotron-3-ultra-550b-a55b:free"   # experimento B
RAZONAMIENTO    = {"enabled": False}   # los modelos :free razonan por defecto; si no se desactiva gastan MAX_TOKENS_SALIDA pensando

TOKENIZADOR_FRAGMENTOS = EMB_PRINCIPAL
TEMPERATURA = 0.0
MAX_TOKENS_SALIDA = 512
MAX_TOKENS_CONTEXTO = 2400
SEMILLA = 42                    # se envía al proveedor; si la ignora, se registra como límite
PAUSA_ENTRE_LLAMADAS_S = 4.0    # los modelos :free de OpenRouter limitan ~20 solicitudes/minuto
MAX_INTENTOS = 4

BASE = dict(estrategia="fija", chunk_size=400, overlap=60, k=4, busqueda="densa",
            emb=EMB_PRINCIPAL, llm=LLM_PRINCIPAL, usa_rag=True, experimento="A")

CONFIGS = {
    "R0":    dict(BASE),
    "R1":    dict(BASE, chunk_size=200, overlap=30),
    "R2":    dict(BASE, k=8),
    "C0":    dict(BASE, usa_rag=False),
    "B_EMB": dict(BASE, emb=EMB_ALTERNATIVO, experimento="B"),
    "B_LLM": dict(BASE, llm=LLM_ALTERNATIVO, experimento="B"),
    "R2_HIBRIDA": dict(BASE, busqueda="hibrida", experimento="opcional"),
}
CONFIGS_A = ["R0", "R1", "R2", "C0"]
CONFIGS_B = ["B_EMB", "B_LLM"]

DIR_CORPUS = RAIZ / "corpus"
DIR_INDICES = RAIZ / "indices"
DIR_RESULTADOS = RAIZ / "resultados"
DIR_EVAL = RAIZ / "evaluacion"
for d in (DIR_INDICES, DIR_RESULTADOS):
    d.mkdir(exist_ok=True)

pd.DataFrame(CONFIGS).T

,estrategia,chunk_size,overlap,k,busqueda,emb,llm,usa_rag,experimento
R0,fija,400,60,4,densa,intfloat/multilingual-e5-small,google/gemma-4-31b-it:free,True,A
R1,fija,200,30,4,densa,intfloat/multilingual-e5-small,google/gemma-4-31b-it:free,True,A
R2,fija,400,60,8,densa,intfloat/multilingual-e5-small,google/gemma-4-31b-it:free,True,A
C0,fija,400,60,4,densa,intfloat/multilingual-e5-small,google/gemma-4-31b-it:free,False,A
B_EMB,fija,400,60,4,densa,BAAI/bge-m3,google/gemma-4-31b-it:free,True,B
B_LLM,fija,400,60,4,densa,intfloat/multilingual-e5-small,nvidia/nemotron-3-ultra-550b-a55b:free,True,B
R2_HIBRIDA,fija,400,60,4,hibrida,intfloat/multilingual-e5-small,google/gemma-4-31b-it:free,True,opcional


## 3. Ingesta (componente A)

Se leen **todos** los archivos de `corpus/` al iniciar. Cada archivo se convierte en **unidades** con un localizador verificable:

| Formato | Unidad | Localizador |
|---|---|---|
| PDF (texto seleccionable) | página | `p. 3` |
| DOCX | párrafo dentro de su sección (estilo *Heading*) | `§ 4. Bloqueo…, párr. 2` |
| MD | bloque (párrafo, lista o tabla) dentro de su sección `#` | `§ Correo y almacenamiento, líneas 40-41` |

In [5]:
from pypdf import PdfReader
from docx import Document as DocxDocument

@dataclass
class Unidad:
    archivo: str
    formato: str
    seccion: str | None
    localizador: str
    texto: str
    pagina: int | None = None
    parrafo: int | None = None
    lineas: tuple | None = None

def _limpiar(t):
    return re.sub(r"[ \t]+", " ", t).strip()

def cargar_pdf(ruta):
    unidades = []
    for i, pagina in enumerate(PdfReader(ruta).pages, start=1):
        texto = _limpiar(pagina.extract_text() or "")
        if texto:
            unidades.append(Unidad(ruta.name, "pdf", None, f"p. {i}", texto, pagina=i))
    if not unidades:
        raise ValueError(f"{ruta.name}: no tiene texto seleccionable (¿PDF escaneado?)")
    return unidades

def cargar_docx(ruta):
    unidades, seccion, n = [], "(inicio)", 0
    for p in DocxDocument(ruta).paragraphs:
        texto = _limpiar(p.text)
        if not texto:
            continue
        estilo = (p.style.name or "").lower()
        if estilo.startswith("heading") or estilo.startswith("título") or estilo == "title":
            seccion, n = texto, 0
            continue
        n += 1
        unidades.append(Unidad(ruta.name, "docx", seccion, f"§ {seccion}, párr. {n}", texto, parrafo=n))
    return unidades

def cargar_md(ruta):
    unidades, seccion = [], "(inicio)"
    lineas = ruta.read_text(encoding="utf-8").splitlines()
    bloque, inicio = [], None

    def cerrar(fin):
        nonlocal bloque, inicio
        if bloque:
            texto = _limpiar("\n".join(bloque))
            unidades.append(Unidad(ruta.name, "md", seccion,
                                   f"§ {seccion}, líneas {inicio}-{fin}", texto, lineas=(inicio, fin)))
        bloque, inicio = [], None

    for num, linea in enumerate(lineas, start=1):
        m = re.match(r"^(#{1,6})\s+(.*)", linea)
        if m:
            cerrar(num - 1)
            seccion = m.group(2).strip()
        elif not linea.strip():
            cerrar(num - 1)
        else:
            if inicio is None:
                inicio = num
            bloque.append(linea)
    cerrar(len(lineas))
    return unidades

CARGADORES = {".pdf": cargar_pdf, ".docx": cargar_docx, ".md": cargar_md}

def cargar_corpus(carpeta=DIR_CORPUS):
    unidades = []
    for ruta in sorted(carpeta.iterdir()):
        if ruta.suffix.lower() in CARGADORES and ruta.name.lower() != "readme.md":  # el README describe el corpus, no es parte de él
            unidades.extend(CARGADORES[ruta.suffix.lower()](ruta))
    return unidades

UNIDADES = cargar_corpus()
resumen = (pd.DataFrame([asdict(u) for u in UNIDADES])
           .groupby(["archivo", "formato"]).agg(unidades=("texto", "size"),
                                                caracteres=("texto", lambda s: s.str.len().sum())))
assert resumen.index.get_level_values("formato").nunique() >= 2 and len(resumen) >= 3, \
    "El enunciado exige ≥3 archivos y ≥2 formatos"
resumen

,,unidades,caracteres
archivo,formato,,
catalogo_servicios_sla.docx,docx,20,3212
faq_mesa_ayuda.md,md,14,3047
guia_equipos_impresion_software.md,md,14,2840
manual_conectividad_vpn.pdf,pdf,4,5253
politica_cuentas_contrasenas.docx,docx,17,3604
procedimiento_incidentes_seguridad.docx,docx,27,4482


## 4. Fragmentación (componente A)

- **Fija (R0, R1):** cada documento se tokeniza completo y se corta en ventanas de `chunk_size` tokens con `overlap` de solapamiento. Gracias a los *offsets* del tokenizador, cada fragmento sabe qué unidades cubre y hereda sus localizadores (p. ej. `p. 2-3` o `§ 4. Bloqueo…, párr. 1-2`).
- **Por sección (alternativa para R1):** `estrategia="seccion"` agrupa por sección/página y solo parte las que superan `chunk_size`.

In [6]:
from transformers import AutoTokenizer
import transformers
transformers.logging.set_verbosity_error()

TOKENIZADOR = AutoTokenizer.from_pretrained(TOKENIZADOR_FRAGMENTOS)

def n_tokens(texto):
    return len(TOKENIZADOR(texto, add_special_tokens=False)["input_ids"])

def combinar_localizadores(unidades):
    """Resume los localizadores de las unidades que cubre un fragmento."""
    u0 = unidades[0]
    if u0.formato == "pdf":
        a, b = unidades[0].pagina, unidades[-1].pagina
        return f"p. {a}" if a == b else f"p. {a}-{b}"
    partes, actual = [], []
    for u in unidades:
        if actual and u.seccion != actual[0].seccion:
            partes.append(actual); actual = []
        actual.append(u)
    partes.append(actual)
    textos = []
    for grupo in partes:
        g0, g1 = grupo[0], grupo[-1]
        if g0.formato == "docx":
            rango = f"párr. {g0.parrafo}" if g0.parrafo == g1.parrafo else f"párr. {g0.parrafo}-{g1.parrafo}"
        else:
            rango = f"líneas {g0.lineas[0]}-{g1.lineas[1]}"
        textos.append(f"§ {g0.seccion}, {rango}")
    return " ; ".join(textos)

def _ventanas(texto, spans, unidades, archivo, formato, chunk_size, overlap, prefijo_id):
    enc = TOKENIZADOR(texto, add_special_tokens=False, return_offsets_mapping=True)
    offsets = enc["offset_mapping"]
    paso = chunk_size - overlap
    fragmentos, inicio = [], 0
    while inicio < len(offsets):
        fin = min(inicio + chunk_size, len(offsets))
        c0, c1 = offsets[inicio][0], offsets[fin - 1][1]
        cubiertas = [u for (a, b), u in zip(spans, unidades) if a < c1 and b > c0]
        fragmentos.append(dict(
            chunk_id=f"{prefijo_id}::c{len(fragmentos) + 1}", archivo=archivo, formato=formato,
            localizador=combinar_localizadores(cubiertas), texto=texto[c0:c1].strip(),
            n_tokens=fin - inicio))
        if fin == len(offsets):
            break
        inicio += paso
    return fragmentos

def fragmentar(unidades, estrategia="fija", chunk_size=400, overlap=60):
    por_archivo = {}
    for u in unidades:
        por_archivo.setdefault(u.archivo, []).append(u)
    fragmentos = []
    for archivo, us in por_archivo.items():
        if estrategia == "fija":
            grupos = [us]
        elif estrategia == "seccion":
            grupos, clave = [], None
            for u in us:
                k = u.pagina if u.formato == "pdf" else u.seccion
                if k != clave:
                    grupos.append([]); clave = k
                grupos[-1].append(u)
        else:
            raise ValueError(estrategia)
        for gi, grupo in enumerate(grupos, start=1):
            texto, spans = "", []
            for u in grupo:
                if texto:
                    texto += "\n\n"
                spans.append((len(texto), len(texto) + len(u.texto)))
                texto += u.texto
            prefijo = archivo if estrategia == "fija" else f"{archivo}::s{gi}"
            fragmentos += _ventanas(texto, spans, grupo, archivo, grupo[0].formato,
                                    chunk_size, overlap, prefijo)
    return fragmentos

for nombre in ["R0", "R1"]:
    c = CONFIGS[nombre]
    fr = fragmentar(UNIDADES, c["estrategia"], c["chunk_size"], c["overlap"])
    print(f"{nombre}: {len(fr)} fragmentos · tokens máx. {max(f['n_tokens'] for f in fr)}")
pd.DataFrame(fragmentar(UNIDADES))[["chunk_id", "localizador", "n_tokens", "texto"]].head(6)

R0: 18 fragmentos · tokens máx. 400
R1: 33 fragmentos · tokens máx. 200


,chunk_id,localizador,n_tokens,texto
0,catalogo_servicios_sla.docx::c1,"§ Catálogo de servicios y acuerdos de nivel de servicio, párr. 1 ; § 1. Introducción, párr. 1 ; § 2. Canales y horar...",400,DOCUMENTO SINTÉTICO - Contenido ficticio creado para el proyecto académico RAG de Sistemas Inteligentes. El Institut...
1,catalogo_servicios_sla.docx::c2,"§ 3. Niveles de prioridad, párr. 4 ; § 4. Tiempos de respuesta y de resolución, párr. 1-5 ; § 5. Escalamiento, párr....",400,"- Baja: solicitudes de servicio y consultas, como instalar software, crear una carpeta compartida o pedir informació..."
2,catalogo_servicios_sla.docx::c3,"§ 6. Servicios del catálogo, párr. 3-4 ; § 7. Encuestas de satisfacción, párr. 1",82,"del jefe de la dependencia propietaria.\n\nAcceso remoto (VPN) para contratistas: P4, con aprobación del supervisor ..."
3,faq_mesa_ayuda.md::c1,"§ Preguntas frecuentes - Mesa de Ayuda de TI, líneas 3-3 ; § Cómo pedir ayuda, líneas 7-18 ; § Clasificación de inci...",400,> DOCUMENTO SINTÉTICO - Contenido ficticio creado para el proyecto académico RAG de Sistemas Inteligentes. El Instit...
4,faq_mesa_ayuda.md::c2,"§ Clasificación de incidentes comunes, líneas 22-31 ; § Cuentas y contraseñas, líneas 35-40 ; § Correo y almacenamie...",400,"Instalación de software | P4 |\n| Solicitud de carpeta compartida | P4 |\n\n**Olvidé mi contraseña, ¿qué hago?**\nCá..."
5,faq_mesa_ayuda.md::c3,"§ Equipos y software, líneas 63-69 ; § Seguridad, líneas 73-75",136,instalar programas en mi equipo?**\nNo. Los equipos institucionales no permiten instalar software sin permisos de ad...


## 5. Embeddings e índice (componente A)

Contrato con el componente B:
- `indexar(config)` → construye (o carga de `indices/`) los fragmentos y la matriz de embeddings normalizados.
- `embeber_consulta(texto, config)` → vector de la consulta con **el mismo modelo** del índice de esa configuración.

La similitud coseno es el producto punto entre vectores normalizados.

In [7]:
from sentence_transformers import SentenceTransformer

PREFIJOS = {  # e5 necesita prefijos; bge-m3 no
    "intfloat/multilingual-e5-small": ("query: ", "passage: "),
    "intfloat/multilingual-e5-base": ("query: ", "passage: "),
}
_MODELOS = {}

def modelo_emb(nombre):
    if nombre not in _MODELOS:
        _MODELOS[nombre] = SentenceTransformer(nombre)
    return _MODELOS[nombre]

def _slug(texto):
    return re.sub(r"[^a-zA-Z0-9]+", "_", texto).strip("_")

def clave_fragmentacion(cfg):
    return f"{cfg['estrategia']}_{cfg['chunk_size']}_{cfg['overlap']}"

@dataclass
class Indice:
    nombre: str
    emb: str
    fragmentos: list
    matriz: np.ndarray
    bm25: object = None

def indexar(cfg, forzar=False):
    ruta = DIR_INDICES / f"{clave_fragmentacion(cfg)}__{_slug(cfg['emb'])}"
    if ruta.exists() and not forzar:
        fragmentos = json.loads((ruta / "fragmentos.json").read_text(encoding="utf-8"))
        matriz = np.load(ruta / "embeddings.npy")
    else:
        fragmentos = fragmentar(UNIDADES, cfg["estrategia"], cfg["chunk_size"], cfg["overlap"])
        _, pref_doc = PREFIJOS.get(cfg["emb"], ("", ""))
        matriz = modelo_emb(cfg["emb"]).encode([pref_doc + f["texto"] for f in fragmentos],
                                               normalize_embeddings=True, batch_size=16,
                                               show_progress_bar=False).astype(np.float32)
        ruta.mkdir(parents=True, exist_ok=True)
        (ruta / "fragmentos.json").write_text(json.dumps(fragmentos, ensure_ascii=False, indent=1), encoding="utf-8")
        np.save(ruta / "embeddings.npy", matriz)
        (ruta / "meta.json").write_text(json.dumps(dict(
            modelo_embeddings=cfg["emb"], tokenizador_fragmentos=TOKENIZADOR_FRAGMENTOS,
            estrategia=cfg["estrategia"], chunk_size=cfg["chunk_size"], overlap=cfg["overlap"],
            n_fragmentos=len(fragmentos), dimension=int(matriz.shape[1]),
            creado=datetime.now(timezone.utc).isoformat()), indent=1), encoding="utf-8")
    return Indice(ruta.name, cfg["emb"], fragmentos, matriz)

def embeber_consulta(texto, cfg):
    pref_q, _ = PREFIJOS.get(cfg["emb"], ("", ""))
    return modelo_emb(cfg["emb"]).encode([pref_q + texto], normalize_embeddings=True)[0].astype(np.float32)

INDICES = {}
for nombre, cfg in CONFIGS.items():
    if cfg["usa_rag"]:
        INDICES[nombre] = indexar(cfg)
        print(f"{nombre:<11} índice {INDICES[nombre].nombre:<55} {INDICES[nombre].matriz.shape}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

R0          índice fija_400_60__intfloat_multilingual_e5_small             (18, 384)


R1          índice fija_200_30__intfloat_multilingual_e5_small             (33, 384)
R2          índice fija_400_60__intfloat_multilingual_e5_small             (18, 384)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

... (9 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

B_EMB       índice fija_400_60__BAAI_bge_m3                                (18, 1024)
B_LLM       índice fija_400_60__intfloat_multilingual_e5_small             (18, 384)
R2_HIBRIDA  índice fija_400_60__intfloat_multilingual_e5_small             (18, 384)


## 6. Recuperación, contexto y LLM (componente B)

- **Densa:** top-k por similitud coseno.
- **Híbrida (opcional):** fusiona el ranking denso con BM25 mediante *Reciprocal Rank Fusion* (RRF, k = 60) y conserva el mismo k.
- **Contexto:** se agregan fragmentos completos en orden de rango mientras quepan en `MAX_TOKENS_CONTEXTO`; los que no caben se registran como recortados (nunca se corta un fragmento a la mitad).

In [8]:
from rank_bm25 import BM25Okapi

def _normalizar(t):
    t = unicodedata.normalize("NFKD", t.lower())
    return "".join(c for c in t if not unicodedata.combining(c))

def _tokens_bm25(t):
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)?", _normalizar(t))

def recuperar(consulta, nombre_cfg):
    """Devuelve [{chunk_id, puntaje, rango, ...fragmento}] ordenados (contrato B)."""
    cfg, idx = CONFIGS[nombre_cfg], INDICES[nombre_cfg]
    sims = idx.matriz @ embeber_consulta(consulta, cfg)
    if cfg["busqueda"] == "densa":
        orden = np.argsort(-sims)[: cfg["k"]]
        puntajes = sims[orden]
    elif cfg["busqueda"] == "hibrida":
        if idx.bm25 is None:
            idx.bm25 = BM25Okapi([_tokens_bm25(f["texto"]) for f in idx.fragmentos])
        bm = idx.bm25.get_scores(_tokens_bm25(consulta))
        rrf = np.zeros(len(sims))
        for ranking in (np.argsort(-sims), np.argsort(-bm)):
            for r, i in enumerate(ranking[:50], start=1):
                rrf[i] += 1.0 / (60 + r)
        orden = np.argsort(-rrf)[: cfg["k"]]
        puntajes = rrf[orden]
    else:
        raise ValueError(cfg["busqueda"])
    return [dict(idx.fragmentos[i], puntaje=float(p), rango=r)
            for r, (i, p) in enumerate(zip(orden, puntajes), start=1)]

def construir_contexto(recuperados, limite=MAX_TOKENS_CONTEXTO):
    enviados, usados = [], 0
    for f in recuperados:
        bloque = f"[F{len(enviados) + 1}] ({f['archivo']} · {f['localizador']})\n{f['texto']}"
        t = n_tokens(bloque)
        if usados + t > limite:
            break
        enviados.append(dict(f, etiqueta=f"F{len(enviados) + 1}", bloque=bloque))
        usados += t
    recortados = [f["chunk_id"] for f in recuperados[len(enviados):]]
    return enviados, recortados, usados

In [9]:
FRASE_ABSTENCION_RAG = "EVIDENCIA INSUFICIENTE"
FRASE_ABSTENCION_C0 = "NO TENGO INFORMACIÓN"

PROMPT_RAG_SISTEMA = f"""Eres el asistente de la Mesa de Ayuda de TI del Instituto Tecnológico Cóndor.
Responde ÚNICAMENTE con la información de los fragmentos de contexto; no uses conocimiento externo.
Después de cada afirmación, cita entre corchetes la etiqueta del fragmento que la respalda, por ejemplo [F2].
Si la respuesta requiere información de varios fragmentos, cita cada uno donde corresponda.
Si los fragmentos no contienen la información necesaria para responder, responde exactamente
"{FRASE_ABSTENCION_RAG}:" seguido de una frase que diga qué información falta, sin dar ninguna respuesta.
Responde en español, de forma breve y directa."""

PROMPT_RAG_USUARIO = """Fragmentos de contexto:

{contexto}

Pregunta: {pregunta}"""

PROMPT_C0_SISTEMA = f"""Eres un asistente de mesa de ayuda de TI.
Responde la pregunta con tu propio conocimiento.
Si no tienes información suficiente o confiable para responder, responde exactamente
"{FRASE_ABSTENCION_C0}:" seguido de una frase que explique por qué, sin inventar datos.
Responde en español, de forma breve y directa."""

# Se guardan para documentar la plantilla exacta usada en las corridas
(DIR_RESULTADOS / "prompts.json").write_text(json.dumps(dict(
    rag_sistema=PROMPT_RAG_SISTEMA, rag_usuario=PROMPT_RAG_USUARIO, c0_sistema=PROMPT_C0_SISTEMA),
    ensure_ascii=False, indent=1), encoding="utf-8")

1070

In [10]:
from openai import OpenAI

CLIENTE = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY or "sin-clave")

def llamar_llm(mensajes, modelo):
    """Una llamada al LLM con los parámetros fijos. Devuelve (texto, modelo_reportado, uso)."""
    r = CLIENTE.chat.completions.create(model=modelo, messages=mensajes, temperature=TEMPERATURA,
                                        max_tokens=MAX_TOKENS_SALIDA, seed=SEMILLA,
                                        extra_body={"reasoning": RAZONAMIENTO})
    if not r.choices or not (r.choices[0].message.content or "").strip():
        raise RuntimeError(f"Respuesta vacía del proveedor (¿se agotó MAX_TOKENS_SALIDA razonando?): {r}")
    uso = r.usage.model_dump() if r.usage else {}
    return r.choices[0].message.content.strip(), r.model, uso

In [11]:
# Verifica que los modelos configurados existan en OpenRouter; si no, sugiere modelos gratuitos.
import urllib.request
try:
    with urllib.request.urlopen("https://openrouter.ai/api/v1/models", timeout=20) as resp:
        _ids = {m["id"] for m in json.load(resp)["data"]}
    for m in (LLM_PRINCIPAL, LLM_ALTERNATIVO):
        print(("✔" if m in _ids else "✘ NO EXISTE"), m)
    if not {LLM_PRINCIPAL, LLM_ALTERNATIVO} <= _ids:
        print("\nModelos gratuitos disponibles (elige dos y actualiza la sección 2):")
        print("\n".join(sorted(i for i in _ids if i.endswith(":free"))))
except Exception as e:
    print("No se pudo consultar la lista de modelos:", e)

✔ google/gemma-4-31b-it:free
✔ nvidia/nemotron-3-ultra-550b-a55b:free


## 7. Asistente: respuesta, fragmentos en orden y citas

`responder(pregunta, config)` ejecuta el flujo completo y devuelve un registro con todo lo que exige el enunciado. `mostrar(registro)` lo presenta: la respuesta, los fragmentos recuperados en orden (indicando cuáles entraron al contexto) y las citas resueltas a **archivo · localizador**.

In [12]:
def _es_abstencion(texto):
    t = _normalizar(texto).lstrip(" *\"'")
    return t.startswith(_normalizar(FRASE_ABSTENCION_RAG)) or t.startswith(_normalizar(FRASE_ABSTENCION_C0))

def responder(pregunta, nombre_cfg):
    cfg = CONFIGS[nombre_cfg]
    t0 = time.perf_counter()
    recuperados, enviados, recortados, tokens_ctx = [], [], [], 0
    if cfg["usa_rag"]:
        recuperados = recuperar(pregunta, nombre_cfg)
        enviados, recortados, tokens_ctx = construir_contexto(recuperados)
        contexto = "\n\n".join(f["bloque"] for f in enviados)
        mensajes = [{"role": "system", "content": PROMPT_RAG_SISTEMA},
                    {"role": "user", "content": PROMPT_RAG_USUARIO.format(contexto=contexto, pregunta=pregunta)}]
    else:
        mensajes = [{"role": "system", "content": PROMPT_C0_SISTEMA},
                    {"role": "user", "content": pregunta}]
    t1 = time.perf_counter()
    respuesta, modelo_reportado, uso = llamar_llm(mensajes, cfg["llm"])
    t2 = time.perf_counter()

    por_etiqueta = {f["etiqueta"]: f for f in enviados}
    etiquetas = list(dict.fromkeys(re.findall(r"\[?\b(F\d+)\b\]?", respuesta))) if cfg["usa_rag"] else []
    citas = [dict(etiqueta=e, chunk_id=por_etiqueta[e]["chunk_id"], archivo=por_etiqueta[e]["archivo"],
                  localizador=por_etiqueta[e]["localizador"]) for e in etiquetas if e in por_etiqueta]
    citas_invalidas = [e for e in etiquetas if e not in por_etiqueta]
    return dict(
        configuracion=nombre_cfg, pregunta=pregunta, respuesta=respuesta,
        abstencion_detectada=_es_abstencion(respuesta),
        recuperados=[{k: f[k] for k in ("chunk_id", "archivo", "localizador", "puntaje", "rango", "texto")} for f in recuperados],
        enviados=[f["chunk_id"] for f in enviados], recortados=recortados, hubo_recorte=bool(recortados),
        tokens_contexto=tokens_ctx, citas=citas, citas_invalidas=citas_invalidas,
        mensajes=mensajes, modelo_reportado=modelo_reportado, uso=uso,
        tiempo_recuperacion_s=round(t1 - t0, 3), tiempo_llm_s=round(t2 - t1, 3), tiempo_s=round(t2 - t0, 3))

def mostrar(reg):
    out = [f"### {reg['configuracion']} · {reg['pregunta']}", "", "**Respuesta:**", "", reg["respuesta"], ""]
    if reg["recuperados"]:
        out += ["**Fragmentos recuperados (en orden):**", "",
                "| # | Puntaje | Archivo | Localizador | ¿En contexto? | Inicio del texto |", "|---|---|---|---|---|---|"]
        for f in reg["recuperados"]:
            dentro = "sí" if f["chunk_id"] in reg["enviados"] else "**recortado**"
            inicio = f["texto"][:90].replace("\n", " ").replace("|", "/")
            out.append(f"| {f['rango']} | {f['puntaje']:.3f} | {f['archivo']} | {f['localizador']} | {dentro} | {inicio}… |")
        out += ["", "**Citas:**", ""]
        out += [f"- [{c['etiqueta']}] {c['archivo']} · {c['localizador']}" for c in reg["citas"]] or ["- (sin citas)"]
        if reg["citas_invalidas"]:
            out.append(f"- ⚠ citas a fragmentos inexistentes: {reg['citas_invalidas']}")
    if reg["abstencion_detectada"]:
        out += ["", "_El sistema indicó que la evidencia es insuficiente._"]
    out += ["", f"_Tiempo: {reg['tiempo_s']} s · modelo: {reg['modelo_reportado']}_"]
    display(Markdown("\n".join(out)))

In [13]:
# Prueba rápida del asistente (gasta 1 llamada al LLM)
if OPENROUTER_API_KEY:
    mostrar(responder("¿Qué hago si la VPN me muestra el error E-412?", "R0"))
else:
    print("Sin clave: se muestra solo la recuperación.")
    display(pd.DataFrame(recuperar("¿Qué hago si la VPN me muestra el error E-412?", "R0"))[["rango", "puntaje", "archivo", "localizador"]])

Sin clave: se muestra solo la recuperación.


,rango,puntaje,archivo,localizador
0,1,0.861832,manual_conectividad_vpn.pdf,p. 3-4
1,2,0.853430,manual_conectividad_vpn.pdf,p. 1-3
2,3,0.841923,faq_mesa_ayuda.md,"§ Preguntas frecuentes - Mesa de Ayuda de TI, líneas 3-3 ; § Cómo pedir ayuda, líneas 7-18 ; § Clasificación de inci..."
3,4,0.829273,politica_cuentas_contrasenas.docx,"§ 3. Autenticación de dos factores (MFA), párr. 2-3 ; § 4. Bloqueo y desbloqueo de cuentas, párr. 1-3 ; § 5. Cuentas..."


## 8. Preguntas de referencia y cobertura de evidencia

Las 9 preguntas (3 directas, 3 multidocumento, 3 sin respuesta) están en `evaluacion/preguntas.json`, con la respuesta esperada y la **evidencia literal** (archivo + localizador + texto). La celda siguiente verifica que cada cita de evidencia exista en el corpus, y muestra la cobertura de evidencia de la recuperación **sin llamar al LLM** (útil para probar R0/R1/R2 antes de gastar llamadas).

Un pasaje está *cubierto* si aparece completo dentro de algún fragmento; si queda partido entre dos fragmentos, no cuenta (justo lo que R1 permite observar).

In [14]:
PREGUNTAS = json.loads((DIR_EVAL / "preguntas.json").read_text(encoding="utf-8"))["preguntas"]
assert len(PREGUNTAS) == 9 and {p["tipo"] for p in PREGUNTAS} == {"directa", "multidocumento", "sin_respuesta"}

def _norm_ev(t):
    return re.sub(r"\s+", " ", _normalizar(t)).strip()

TEXTO_POR_ARCHIVO = {}
for u in UNIDADES:
    TEXTO_POR_ARCHIVO[u.archivo] = TEXTO_POR_ARCHIVO.get(u.archivo, "") + " " + _norm_ev(u.texto)
for p in PREGUNTAS:
    for ev in p["evidencia"]:
        assert _norm_ev(ev["texto"]) in TEXTO_POR_ARCHIVO.get(ev["archivo"], ""), \
            f"{p['id']}: la evidencia no está en {ev['archivo']}: {ev['texto'][:60]}"
    if p["tipo"] == "multidocumento":
        assert len({ev["archivo"] for ev in p["evidencia"]}) >= 2, f"{p['id']} debe usar ≥2 documentos"
print("✔ Toda la evidencia existe en el corpus; las multidocumento usan ≥2 archivos.")

def cobertura(pregunta, chunk_ids, nombre_cfg):
    """1 si todos los pasajes necesarios están dentro de algún fragmento de chunk_ids; None si no aplica."""
    if not pregunta["evidencia"]:
        return None
    textos = {f["chunk_id"]: _norm_ev(f["texto"]) for f in INDICES[nombre_cfg].fragmentos}
    return int(all(any(_norm_ev(ev["texto"]) in textos[c] for c in chunk_ids) for ev in pregunta["evidencia"]))

filas = []
for nombre in ["R0", "R1", "R2", "B_EMB", "R2_HIBRIDA"]:
    for p in PREGUNTAS:
        if p["evidencia"]:
            rec = recuperar(p["pregunta"], nombre)
            env, _, _ = construir_contexto(rec)
            filas.append(dict(config=nombre, pregunta=p["id"],
                              recuperado=cobertura(p, [f["chunk_id"] for f in rec], nombre),
                              contexto=cobertura(p, [f["chunk_id"] for f in env], nombre)))
cob = pd.DataFrame(filas)
cob.pivot_table(index="pregunta", columns="config", values=["recuperado", "contexto"]).astype(int)

✔ Toda la evidencia existe en el corpus; las multidocumento usan ≥2 archivos.


contexto                     recuperado                    
config      B_EMB R0 R1 R2 R2_HIBRIDA      B_EMB R0 R1 R2 R2_HIBRIDA
pregunta                                                            
D1              1  1  1  1          1          1  1  1  1          1
D2              1  1  1  1          1          1  1  1  1          1
D3              1  1  1  1          1          1  1  1  1          1
M1              0  1  1  1          1          0  1  1  1          1
M2              0  1  0  1          1          0  1  0  1          1
M3              0  1  0  1          1          0  1  0  1          1

## 9. Corridas finales

> ⚠ **Antes de ejecutar:** las 9 preguntas y las hipótesis (`plantillas/hipotesis.md`) deben estar congeladas con un commit fechado.

- Experimento A: R0, R1, R2 y C0 × 9 preguntas = **36 corridas**. Experimento B: B_EMB y B_LLM × 9 = **18 corridas**.
- Cada corrida se guarda en `resultados/corridas.jsonl` (detalle completo, incluido el contexto enviado). Si una llamada falla, el fallo se registra en `resultados/fallos.jsonl` y se repite con la misma configuración (hasta `MAX_INTENTOS`).
- Es **reanudable**: si se corta (p. ej. por límite de solicitudes), al volver a ejecutar solo hace las corridas que faltan.

In [15]:
EJECUTAR_CORRIDAS = False          # ← poner True para las corridas finales
CONFIGS_A_EJECUTAR = CONFIGS_A + CONFIGS_B

ARCHIVO_CORRIDAS = DIR_RESULTADOS / "corridas.jsonl"
ARCHIVO_FALLOS = DIR_RESULTADOS / "fallos.jsonl"

def _leer_jsonl(ruta):
    return [json.loads(l) for l in ruta.read_text(encoding="utf-8").splitlines() if l.strip()] if ruta.exists() else []

def _anexar(ruta, obj):
    with ruta.open("a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def ejecutar_corridas(configs, preguntas=PREGUNTAS):
    hechas = {(r["configuracion"], r["pregunta_id"]) for r in _leer_jsonl(ARCHIVO_CORRIDAS)}
    pendientes = [(c, p) for c in configs for p in preguntas if (c, p["id"]) not in hechas]
    print(f"Pendientes: {len(pendientes)} (ya hechas: {len(hechas)})")
    for n, (nombre_cfg, p) in enumerate(pendientes, start=1):
        cfg = CONFIGS[nombre_cfg]
        for intento in range(1, MAX_INTENTOS + 1):
            try:
                reg = responder(p["pregunta"], nombre_cfg)
                break
            except Exception as e:
                _anexar(ARCHIVO_FALLOS, dict(timestamp=datetime.now(timezone.utc).isoformat(),
                                              configuracion=nombre_cfg, pregunta_id=p["id"],
                                              intento=intento, error=f"{type(e).__name__}: {e}"[:500]))
                print(f"  fallo {nombre_cfg}/{p['id']} intento {intento}: {type(e).__name__}")
                time.sleep(PAUSA_ENTRE_LLAMADAS_S * 3 * intento)
        else:
            print(f"  ✘ {nombre_cfg}/{p['id']} sin éxito tras {MAX_INTENTOS} intentos; vuelve a ejecutar más tarde")
            continue
        reg.update(
            run_id=hashlib.sha1(f"{nombre_cfg}|{p['id']}|{time.time()}".encode()).hexdigest()[:10],
            timestamp=datetime.now(timezone.utc).isoformat(), pregunta_id=p["id"], tipo_pregunta=p["tipo"],
            experimento=cfg["experimento"], intento=intento,
            parametros=dict(cfg, tokenizador=TOKENIZADOR_FRAGMENTOS, temperatura=TEMPERATURA,
                            max_tokens_salida=MAX_TOKENS_SALIDA, max_tokens_contexto=MAX_TOKENS_CONTEXTO, semilla=SEMILLA,
                            razonamiento=RAZONAMIENTO),
            cobertura_recuperado=cobertura(p, [f["chunk_id"] for f in reg["recuperados"]], nombre_cfg) if cfg["usa_rag"] else None,
            cobertura_contexto=cobertura(p, reg["enviados"], nombre_cfg) if cfg["usa_rag"] else None)
        _anexar(ARCHIVO_CORRIDAS, reg)
        print(f"[{n}/{len(pendientes)}] {nombre_cfg}/{p['id']} ✔ {reg['tiempo_s']} s" +
              (" · abstención" if reg["abstencion_detectada"] else ""))
        time.sleep(PAUSA_ENTRE_LLAMADAS_S)

if EJECUTAR_CORRIDAS:
    assert OPENROUTER_API_KEY, "Falta OPENROUTER_API_KEY"
    ejecutar_corridas(CONFIGS_A_EJECUTAR)
else:
    print("EJECUTAR_CORRIDAS = False → no se llamó al LLM. Corridas registradas:", len(_leer_jsonl(ARCHIVO_CORRIDAS)))

EJECUTAR_CORRIDAS = False → no se llamó al LLM. Corridas registradas: 0


In [16]:
# Registro tabular (formato de plantillas/registro_corridas.csv) + plantilla de valoración para los evaluadores
CORRIDAS = _leer_jsonl(ARCHIVO_CORRIDAS)

def _fila_registro(r):
    p = r["parametros"]
    return dict(
        run_id=r["run_id"], timestamp=r["timestamp"], configuracion=r["configuracion"], experimento=r["experimento"],
        pregunta_id=r["pregunta_id"], tipo_pregunta=r["tipo_pregunta"], modelo_llm=p["llm"],
        modelo_llm_reportado=r["modelo_reportado"], modelo_embeddings=p["emb"] if p["usa_rag"] else "",
        chunk_size=p["chunk_size"], overlap=p["overlap"], unidad=f"tokens ({p['tokenizador']})", k=p["k"],
        tipo_busqueda=p["busqueda"] if p["usa_rag"] else "sin recuperación", temperatura=p["temperatura"],
        max_tokens_salida=p["max_tokens_salida"], max_tokens_contexto=p["max_tokens_contexto"],
        fragmentos_recuperados=" | ".join(f"{f['rango']}:{f['chunk_id']}" for f in r["recuperados"]),
        fragmentos_en_contexto=" | ".join(r["enviados"]), hubo_recorte=r["hubo_recorte"],
        respuesta=r["respuesta"], citas=" | ".join(f"{c['archivo']} · {c['localizador']}" for c in r["citas"]),
        tiempo_s=r["tiempo_s"], intento=r["intento"], abstencion_detectada=r["abstencion_detectada"],
        cobertura_recuperado=r["cobertura_recuperado"], cobertura_contexto=r["cobertura_contexto"])

if CORRIDAS:
    REGISTRO = pd.DataFrame([_fila_registro(r) for r in CORRIDAS]).sort_values(["experimento", "configuracion", "pregunta_id"])
    REGISTRO.to_csv(DIR_RESULTADOS / "registro_corridas.csv", index=False)
    plantilla = REGISTRO[["run_id", "configuracion", "pregunta_id", "tipo_pregunta", "respuesta", "citas",
                          "abstencion_detectada"]].copy()
    for col in ["correcta", "sustantiva", "citas_respaldan", "abstencion_apropiada", "notas"]:
        plantilla[col] = ""
    ruta_plantilla = DIR_EVAL / "valoracion_plantilla.csv"
    plantilla.to_csv(ruta_plantilla, index=False)
    print(f"{len(REGISTRO)} corridas → resultados/registro_corridas.csv; plantilla de valoración → {ruta_plantilla.name}")
    display(REGISTRO.groupby(["experimento", "configuracion"]).size().rename("corridas"))
else:
    print("Aún no hay corridas.")

Aún no hay corridas.


## 10. Valoración y métricas

1. Dos evaluadores copian `evaluacion/valoracion_plantilla.csv` como `valoracion_e1.csv` y `valoracion_e2.csv` y llenan, **sin verse**, las columnas con 1/0 según `plantillas/rubrica_evaluacion.md`:
   - `correcta` (respondibles D/M: correcta **y completa**; parcial o abstención = 0)
   - `sustantiva` (la respuesta intenta contestar con al menos una afirmación)
   - `citas_respaldan` (solo RAG y si es sustantiva: las afirmaciones centrales están respaldadas por los pasajes **citados**)
   - `abstencion_apropiada` (sin respuesta N: reconoce que no puede responder **y** no añade una respuesta sin respaldo)
2. La celda siguiente muestra los desacuerdos. Resuélvanlos y guarden el consenso como `valoracion_final.csv`.
3. Las métricas se calculan desde `valoracion_final.csv` (la cobertura se calcula sola a partir de la evidencia).

In [17]:
COLS_VAL = ["correcta", "sustantiva", "citas_respaldan", "abstencion_apropiada"]
e1, e2 = DIR_EVAL / "valoracion_e1.csv", DIR_EVAL / "valoracion_e2.csv"
if e1.exists() and e2.exists():
    v1, v2 = pd.read_csv(e1).set_index("run_id"), pd.read_csv(e2).set_index("run_id")
    desacuerdos = []
    for col in COLS_VAL:
        dif = v1[col].fillna(-1).astype(int) != v2[col].reindex(v1.index).fillna(-1).astype(int)
        for rid in v1.index[dif]:
            desacuerdos.append(dict(run_id=rid, config=v1.loc[rid, "configuracion"], pregunta=v1.loc[rid, "pregunta_id"],
                                    columna=col, e1=v1.loc[rid, col], e2=v2.loc[rid, col]))
    print(f"Desacuerdos: {len(desacuerdos)}")
    display(pd.DataFrame(desacuerdos))
else:
    print("Faltan valoracion_e1.csv / valoracion_e2.csv")

Faltan valoracion_e1.csv / valoracion_e2.csv


In [18]:
def calcular_metricas(valoracion, registro):
    df = registro.merge(valoracion[["run_id"] + COLS_VAL], on="run_id", how="left")
    filas = []
    for nombre, g in df.groupby("configuracion", sort=False):
        es_rag = CONFIGS[nombre]["usa_rag"]
        resp = g[g.tipo_pregunta != "sin_respuesta"]
        sinr = g[g.tipo_pregunta == "sin_respuesta"]
        sust = g[g.sustantiva == 1]
        filas.append({
            "configuración": nombre, "experimento": CONFIGS[nombre]["experimento"],
            "cobertura recuperado (/6)": f"{int(resp.cobertura_recuperado.sum())}/{len(resp)}" if es_rag else "—",
            "cobertura contexto (/6)": f"{int(resp.cobertura_contexto.sum())}/{len(resp)}" if es_rag else "—",
            "corrección (/6)": f"{int(resp.correcta.fillna(0).sum())}/{len(resp)}",
            "respaldo de citas": ("no aplica" if len(sust) == 0 else
                                  f"{sust.citas_respaldan.fillna(0).sum() / len(sust):.0%} ({int(sust.citas_respaldan.fillna(0).sum())}/{len(sust)})")
                                 if es_rag else "— (C0 sin citas)",
            "abstención apropiada (/3)": f"{int(sinr.abstencion_apropiada.fillna(0).sum())}/{len(sinr)}",
            "recortes de contexto": int(g.hubo_recorte.sum()) if es_rag else "—",
            "tiempo medio (s)": round(g.tiempo_s.mean(), 2),
        })
    orden = [c for c in CONFIGS if c in df.configuracion.unique()]
    return pd.DataFrame(filas).set_index("configuración").loc[orden]

ruta_final = DIR_EVAL / "valoracion_final.csv"
if ruta_final.exists() and CORRIDAS:
    METRICAS = calcular_metricas(pd.read_csv(ruta_final), REGISTRO)
    METRICAS.to_csv(DIR_RESULTADOS / "tabla_resultados.csv")
    display(METRICAS)
else:
    print("Falta evaluacion/valoracion_final.csv (o no hay corridas).")

Falta evaluacion/valoracion_final.csv (o no hay corridas).


## 11. Entorno y parámetros efectivos

Se guarda en `resultados/entorno.json` para el informe. **Límites de la comparación:** los modelos `:free` de OpenRouter pueden ser servidos por distintos proveedores y no todos respetan `seed`, así que las respuestas pueden variar entre ejecuciones aunque la temperatura sea 0; cada corrida registra el `modelo_reportado` por la API.

In [19]:
import importlib.metadata as md_
entorno = dict(
    fecha=datetime.now(timezone.utc).isoformat(), python=platform.python_version(), plataforma=platform.platform(),
    colab=EN_COLAB, gpu=__import__("torch").cuda.is_available(),
    paquetes={p: md_.version(p) for p in ["sentence-transformers", "transformers", "torch", "pypdf",
                                          "python-docx", "rank-bm25", "openai", "numpy", "pandas"]},
    modelos=dict(emb_principal=EMB_PRINCIPAL, emb_alternativo=EMB_ALTERNATIVO,
                 llm_principal=LLM_PRINCIPAL, llm_alternativo=LLM_ALTERNATIVO, proveedor_llm="OpenRouter"),
    parametros_fijos=dict(tokenizador_fragmentos=TOKENIZADOR_FRAGMENTOS, temperatura=TEMPERATURA,
                          max_tokens_salida=MAX_TOKENS_SALIDA, max_tokens_contexto=MAX_TOKENS_CONTEXTO,
                          semilla=SEMILLA, razonamiento=RAZONAMIENTO, similitud="coseno (producto punto de vectores normalizados)"),
    configuraciones=CONFIGS,
    indices={n: dict(nombre=i.nombre, fragmentos=len(i.fragmentos), dimension=int(i.matriz.shape[1]))
             for n, i in INDICES.items()},
    corpus=resumen.reset_index().to_dict(orient="records"),
    limites=["seed no garantizado por todos los proveedores de OpenRouter",
             "una sola corrida por pregunta y configuración"])
(DIR_RESULTADOS / "entorno.json").write_text(json.dumps(entorno, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
print(json.dumps({k: entorno[k] for k in ("python", "colab", "gpu", "paquetes", "modelos")}, indent=1))

{
 "python": "3.14.10",
 "colab": false,
 "gpu": false,
 "paquetes": {
  "sentence-transformers": "6.3.0",
  "transformers": "5.23.0",
  "torch": "2.15.0+cpu",
  "pypdf": "7.1.0",
  "python-docx": "1.2.0",
  "rank-bm25": "0.2.2",
  "openai": "3.52.0",
  "numpy": "2.5.5",
  "pandas": "3.1.1"
 },
 "modelos": {
  "emb_principal": "intfloat/multilingual-e5-small",
  "emb_alternativo": "BAAI/bge-m3",
  "llm_principal": "google/gemma-4-31b-it:free",
  "llm_alternativo": "nvidia/nemotron-3-ultra-550b-a55b:free",
  "proveedor_llm": "OpenRouter"
 }
}
